# Qwen Master Pipeline: End-to-End Scientific Steering & Representation Analysis

This notebook implements the master orchestration pipeline for extracting, analyzing, and visualizing scientific hypothesis-to-conclusion representations using the **Qwen2.5** language model family.

---

### Pipeline Workflow:
1. **Parameter Configuration (Cell 1):** Centralized control panel defining dataset sources, Qwen model checkpoints, layer ranges, embedding modes, and visualization settings.
2. **Raw Dataset Ingestion:** Ingest raw scraped arXiv papers or extracted paper text.
3. **Semantic Preprocessing (`pp.py`):** Clean and normalize text using conservative scientific filters (retaining negation, capitalization, mathematical semantics) and extract paired **Hypothesis ($X$)** and **Conclusion ($Y$)** statements.
4. **Representation Embedding Extraction:** Compute either:
   - **Difference Embeddings ($\\Delta v$):** Directional transition vectors $\\Delta v = h(\\text{Conclusion}) - h(\\text{Hypothesis})$ across transformer layers.
   - **Single Embeddings:** Individual hidden activation vectors $h(X)$ and $h(Y)$ at target layers.
5. **Statistical Latent Space Profiling:** Compute directional coherence, L2 norm distributions, and centroid shifts across layers.
6. **Dimensionality Reduction & Visualization (`pca.py` / `pca_UMAP.py`):**
   - 2D Layer-by-layer trajectory projections with layer annotations.
   - 3D Latent trajectory plots (PC1, PC2, Layer index).
   - Interactive 3D Plotly visualizations rotatable in-browser.

In [ ]:
# =====================================================================
# CELL 1: MASTER PARAMETER CONFIGURATION
# Set all pipeline constants, model choices, and file paths here.
# =====================================================================

# ---------------------------------------------------------------------
# 1. Dataset & File Paths
# ---------------------------------------------------------------------
# Name or relative path to the raw input dataset.
# Available options in repository:
#   - 'arxiv_extracted.csv'       (Contains extracted hypothesis & conclusion)
#   - 'arxiv_extracted_final.csv' (Curated hypothesis & conclusion set)
#   - 'arxiv_papers.csv'          (Raw scraped paper texts with sections)
RAW_DATASET_NAME = "arxiv_extracted.csv"

# Base output directory for processed data, embeddings, and plots
OUTPUT_BASE_DIR = "output"

# ---------------------------------------------------------------------
# 2. Preprocessing Configuration (pp.py)
# ---------------------------------------------------------------------
HYPOTHESIS_COL = "hypothesis"          # Column name for Hypothesis / Anchor
CONCLUSION_COL = "conclusion"          # Column name for Conclusion / Positive
MIN_HYPOTHESIS_WORDS = 8               # Minimum word threshold for valid hypotheses
MIN_CONCLUSION_WORDS = 15              # Minimum word threshold for valid conclusions
ALLOW_HYPOTHESIS_FALLBACK = True       # Derive candidate from intro/abstract if missing

# ---------------------------------------------------------------------
# 3. Model Architecture & Compute Selection
# ---------------------------------------------------------------------
# Select from Qwen2.5-Instruct checkpoints:
#   - 'Qwen/Qwen2.5-0.5B-Instruct' (Ultra-lightweight / Fast CPU or Low VRAM)
#   - 'Qwen/Qwen2.5-1.5B-Instruct' (Fast experimentation)
#   - 'Qwen/Qwen2.5-3B-Instruct'   (Balanced capacity)
#   - 'Qwen/Qwen2.5-7B-Instruct'   (High accuracy / Standard baseline)
#   - 'Qwen/Qwen2.5-14B-Instruct'  (High capacity / Multi-GPU recommended)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

# Compute device: None (auto-detects 'cuda' if GPU available, else 'cpu')
DEVICE = None

# Torch precision: None (auto: bfloat16 on CUDA, float32 on CPU)
TORCH_DTYPE = None

# ---------------------------------------------------------------------
# 4. Embedding Pipeline Configuration
# ---------------------------------------------------------------------
# Mode options:
#   - 'diff'   : Computes directional difference vectors (h_conclusion - h_hypothesis)
#   - 'single' : Computes individual hypothesis and conclusion hidden activations
EMBEDDING_MODE = "diff"  # Options: 'diff' | 'single'

# Target layer span to extract hidden states across:
#   - tuple: (start_layer, end_layer) e.g., (8, 14)
#   - list of layer indices: [6, 8, 10, 12, 14]
#   - None: automatically selects the middle 6 layers based on total model depth
LAYER_RANGE = (8, 14)

# Subset size for embedding calculation:
# Set to an integer (e.g. 5, 10) for rapid prototyping and interactive testing,
# or set to None to process all clean pairs in the dataset.
MAX_PAIRS_TO_PROCESS = 5

# ---------------------------------------------------------------------
# 5. Latent Space Visualization Settings
# ---------------------------------------------------------------------
# Dimensionality reduction method: 'pca' or 'umap'
VISUALIZATION_METHOD = "pca"

# Number of distinct paper pairs to display in trajectory plots
MAX_PAIRS_TO_PLOT = 5

# Plot export options
SAVE_PLOTS = True   # Save PNG and interactive HTML plots to disk
SHOW_PLOTS = True   # Display plots inline in notebook

## Stage 0: Environment Setup & Dynamic Path Resolution

This cell dynamically detects the repository root (whether executed from `notebooks/` or the repository root), updates `sys.path`, creates the required output directories, and checks available compute hardware.

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

# Dynamically locate repository root and add to sys.path
NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Define and initialize output directories
DATA_PATH = REPO_ROOT / RAW_DATASET_NAME
PREPROCESS_OUT_DIR = REPO_ROOT / OUTPUT_BASE_DIR / "preprocessing"
EMBEDDINGS_OUT_DIR = REPO_ROOT / OUTPUT_BASE_DIR / "embeddings"
VIS_OUT_DIR = REPO_ROOT / OUTPUT_BASE_DIR / "visualizations"

for directory in [PREPROCESS_OUT_DIR, EMBEDDINGS_OUT_DIR, VIS_OUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Determine hardware execution device
cuda_available = torch.cuda.is_available()
device_name = DEVICE if DEVICE else ("cuda" if cuda_available else "cpu")
gpu_info = torch.cuda.get_device_name(0) if cuda_available else "CPU (No GPU detected)"

print("=" * 65)
print(" MASTER PIPELINE ENVIRONMENT INITIALIZED")
print("=" * 65)
print(f"Repository Root:   {REPO_ROOT}")
print(f"Raw Data Target:   {DATA_PATH}")
print(f"Compute Device:    {device_name} ({gpu_info})")
print(f"Selected Model:    {MODEL_ID}")
print(f"Embedding Mode:    {EMBEDDING_MODE.upper()}")
print(f"Layer Range:       {LAYER_RANGE}")
print(f"Max Pairs Sample:  {MAX_PAIRS_TO_PROCESS}")
print("=" * 65)

## Stage 1: Load Raw Dataset

Load the configured raw dataset into memory, inspect available columns, and review the initial rows before preprocessing.

In [ ]:
print(f"Loading raw dataset from: {DATA_PATH}")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at '{DATA_PATH}'. "
        f"Please check RAW_DATASET_NAME in Cell 1."
    )

raw_df = pd.read_csv(DATA_PATH)
print(f"Successfully loaded raw dataset.")
print(f"  • Total Records: {len(raw_df)}")
print(f"  • Total Columns: {len(raw_df.columns)}")
print(f"  • Columns:       {list(raw_df.columns)}")

# Display sample preview
raw_df.head(3)

## Stage 2: Semantic Preprocessing via `pp.py`

We now import the preprocessing module (`pp.py` / `PreProcessing.PP`) and execute conservative scientific filtering on the raw dataset:
- Cleans extraction noise, dehyphenates PDF line breaks, and normalizes LaTeX tags.
- Retains casing, negation, numbers, and mathematical terminology critical to scientific semantics.
- Filters pairs by minimum word count thresholds.
- Deduplicates identical $(X, Y)$ pairs.
- Outputs a clean dataset, a rejection audit log with reasons, and a JSON report.

In [ ]:
# Import the preprocessing script pp.py
try:
    import pp
except ImportError:
    import PreProcessing.PP as pp

print("Running preprocessing script on raw dataset...")
clean_df, rejected_df, report = pp.build_xy_dataset(
    df=raw_df,
    hypothesis_col=HYPOTHESIS_COL,
    conclusion_col=CONCLUSION_COL,
    min_hypothesis_words=MIN_HYPOTHESIS_WORDS,
    min_conclusion_words=MIN_CONCLUSION_WORDS,
    allow_hypothesis_fallback=ALLOW_HYPOTHESIS_FALLBACK,
)

# Export preprocessing outputs
clean_csv_path = PREPROCESS_OUT_DIR / "hypothesis_conclusion_clean.csv"
rejected_csv_path = PREPROCESS_OUT_DIR / "hypothesis_conclusion_rejected.csv"
report_json_path = PREPROCESS_OUT_DIR / "preprocessing_report.json"

clean_df.to_csv(clean_csv_path, index=False)
rejected_df.to_csv(rejected_csv_path, index=False)
report_json_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n" + "=" * 50)
print(" PREPROCESSING SUMMARY REPORT")
print("=" * 50)
print(f"Input Rows:              {report.get('input_rows', 0)}")
print(f"Clean Output (XY) Rows:  {report.get('output_rows', 0)}")
print(f"Rejected Rows:           {report.get('rejected_rows', 0)}")
print(f"Duplicates Removed:      {report.get('duplicate_pairs_removed', 0)}")
if 'mean_hypothesis_words' in report:
    print(f"Mean Hypothesis Words:   {report['mean_hypothesis_words']}")
if 'mean_conclusion_words' in report:
    print(f"Mean Conclusion Words:   {report['mean_conclusion_words']}")
print(f"\nArtifacts saved to: {PREPROCESS_OUT_DIR}")

# Preview cleaned pairs
display_cols = [c for c in ["paper_id", "title", "X_hypothesis", "Y_conclusion", "hypothesis_words", "conclusion_words"] if c in clean_df.columns]
clean_df[display_cols].head(3)

## Stage 3: Representation Embedding Extraction (Diffs vs. Single)

Based on the `EMBEDDING_MODE` setting in Cell 1, this step extracts hidden states from the target Qwen model across the specified layer range:

- **Difference Embeddings (`diff`):** Extracts hidden state activations for both hypothesis and conclusion at each layer, and computes the directional vector difference:
  $$\\Delta v_{\\ell} = h_{\\ell}(\\text{Conclusion}) - h_{\\ell}(\\text{Hypothesis})$$
  This captures the latent transformation vector required to steer the model from premises to conclusions.

- **Single Embeddings (`single`):** Extracts independent activation vectors $h_{\\ell}(X)$ and $h_{\\ell}(Y)$ for each layer into separate dataframes.

In [ ]:
# Prepare dataset slice for embedding
if MAX_PAIRS_TO_PROCESS is not None and MAX_PAIRS_TO_PROCESS < len(clean_df):
    df_to_embed = clean_df.head(MAX_PAIRS_TO_PROCESS).copy()
    print(f"Using subset of {MAX_PAIRS_TO_PROCESS} pairs for embedding extraction.")
else:
    df_to_embed = clean_df.copy()
    print(f"Processing all {len(df_to_embed)} clean pairs for embedding extraction.")

if EMBEDDING_MODE.lower() == "diff":
    print(f"\n>>> Executing Qwen Difference Embeddings Pipeline ({MODEL_ID})...")
    from EmbeddingPipelines.DiffEmbeddings.CalculateQwenEmbeddedDifferents import calculate_qwen_diff_embeddings

    diff_output_csv = EMBEDDINGS_OUT_DIR / "qwen_vector_diffs.csv"
    embeddings_df = calculate_qwen_diff_embeddings(
        data=df_to_embed,
        model_name_or_id=MODEL_ID,
        layer_range=LAYER_RANGE,
        output_path=str(diff_output_csv),
        device=device_name,
        torch_dtype=TORCH_DTYPE,
    )
    print(f"Difference embeddings computed successfully. Shape: {embeddings_df.shape}")

elif EMBEDDING_MODE.lower() == "single":
    print(f"\n>>> Executing Qwen Single Embeddings Pipeline ({MODEL_ID})...")
    from EmbeddingPipelines.SingleEmbeddings.CalculateQwenSingleEmbeddings import calculate_qwen_single_embeddings

    hyp_output_csv = EMBEDDINGS_OUT_DIR / "qwen_single_hyp.csv"
    con_output_csv = EMBEDDINGS_OUT_DIR / "qwen_single_con.csv"

    hyp_df, con_df = calculate_qwen_single_embeddings(
        data=df_to_embed,
        model_name_or_id=MODEL_ID,
        layer_range=LAYER_RANGE,
        hyp_output_path=str(hyp_output_csv),
        con_output_path=str(con_output_csv),
        device=device_name,
        torch_dtype=TORCH_DTYPE,
    )
    print(f"Single embeddings computed: Hypothesis shape {hyp_df.shape}, Conclusion shape {con_df.shape}")

    # Compute difference vectors for downstream trajectory visualization
    dim_cols = [c for c in hyp_df.columns if c.startswith("dim_")]
    embeddings_df = hyp_df[["Layer", "Pair_Index"]].copy()
    if "Paper_ID" in hyp_df.columns:
        embeddings_df["Paper_ID"] = hyp_df["Paper_ID"]
    embeddings_df["Hypothesis"] = hyp_df["x_Hypothesis"]
    embeddings_df["Conclusion"] = con_df["y_conclusion"]
    
    # Vector difference: delta_v = h_conclusion - h_hypothesis
    for c in dim_cols:
        embeddings_df[c] = con_df[c] - hyp_df[c]

    diff_output_csv = EMBEDDINGS_OUT_DIR / "qwen_vector_diffs_from_single.csv"
    embeddings_df.to_csv(diff_output_csv, index=False)
    print(f"Derived vector differences for trajectory visualization saved to: {diff_output_csv}")
else:
    raise ValueError(f"Invalid EMBEDDING_MODE '{EMBEDDING_MODE}'. Must be 'diff' or 'single'.")

# Preview extracted representation table
dim_cols = [c for c in embeddings_df.columns if c.startswith("dim_")]
preview_cols = ["Layer", "Pair_Index"] + dim_cols[:5]
print(f"\nPreview of extracted latent representations (Total dimensions: {len(dim_cols)}):")
embeddings_df[preview_cols].head(6)

## Stage 4: Statistical Profiling of Latent Space Across Layers

To analyze how representations evolve through the transformer's depth, we compute key geometry metrics across layers:
- **Mean L2 Norm ($||\\Delta v||$):** Magnitude of representation shift between hypothesis and conclusion.
- **Centroid Norm:** Magnitude of the average translation direction across all sample pairs.
- **Directional Coherence:** Rayleigh mean resultant length measuring whether hypothesis-to-conclusion vectors point in a consistent direction across papers.
- **Mean Pairwise Cosine Similarity:** Average angular agreement between different paper difference vectors.

In [ ]:
from EmbeddingPipelines.DiffEmbeddings.SummarizeEmbeddings import compute_layer_statistics, print_formatted_report

dim_cols = [c for c in embeddings_df.columns if c.startswith("dim_")]
stats_df = compute_layer_statistics(embeddings_df, dim_cols)

# Export statistics summary
stats_summary_path = EMBEDDINGS_OUT_DIR / "layer_statistics_summary.csv"
stats_df.to_csv(stats_summary_path, index=False)

# Print formatted report table
print_formatted_report(stats_df, str(diff_output_csv))

# Display DataFrame in notebook
stats_df

## Stage 5: Latent Space Dimensionality Reduction & Trajectory Visualization

Now we pass the computed embeddings into the visualization script (`visualization/pca.py`).

The visualization engine produces:
1. **2D PCA Projection Plot:** Displays each scientific paper pair's vector trajectory from lower to higher transformer layers, labeled by layer number ($L_8, L_9, \\dots$).
2. **3D Matplotlib Trajectory Plot:** Adds the model layer index as the Z-axis, visualizing how the semantic shift moves through network depth.
3. **Interactive 3D Plotly Visualization:** A fully interactive, rotatable 3D scene embedded in the notebook and saved as an HTML artifact.

In [ ]:
from visualization.pca import run_pca_visualization

print(f"Passing embeddings into PCA visualization script (Visualizing {MAX_PAIRS_TO_PLOT} pairs)...")

df_pca, pca_model, fig_2d, fig_3d_mpl, fig_3d_plotly = run_pca_visualization(
    data=embeddings_df,
    output_dir=VIS_OUT_DIR,
    n_components=2,
    pairs_to_plot=MAX_PAIRS_TO_PLOT,
    save_plots=SAVE_PLOTS,
    show_plots=SHOW_PLOTS,
)

# Display the interactive Plotly 3D trajectory directly in the notebook
fig_3d_plotly.show()

## Stage 6: Optional Non-Linear Trajectory Visualization (UMAP)

If `VISUALIZATION_METHOD == 'umap'`, this step runs PCA pre-reduction down to 50 dimensions followed by 2D UMAP non-linear manifold projection via `visualization/pca_UMAP.py`.

In [ ]:
if VISUALIZATION_METHOD.lower() in ["umap", "pca_umap"]:
    try:
        from visualization.pca_UMAP import run_umap_visualization
        print("Passing embeddings into UMAP visualization script...")
        df_umap, fig_2d_u, fig_3d_u, fig_3d_plotly_u = run_umap_visualization(
            data=embeddings_df,
            output_dir=VIS_OUT_DIR,
            pairs_to_plot=MAX_PAIRS_TO_PLOT,
            save_plots=SAVE_PLOTS,
            show_plots=SHOW_PLOTS,
        )
        fig_3d_plotly_u.show()
    except ImportError as e:
        print(f"UMAP visualization skipped: {e}")
    except Exception as err:
        print(f"UMAP visualization encountered an error: {err}")
else:
    print(f"UMAP visualization skipped (VISUALIZATION_METHOD is set to '{VISUALIZATION_METHOD}').")

## Stage 7: Pipeline Completion & Generated Artifact Manifest

All stages of the master pipeline have executed. Below is a manifest of all persisted artifacts across the pipeline lifecycle.

In [ ]:
print("=" * 75)
print(" QWEN MASTER PIPELINE ARTIFACT MANIFEST")
print("=" * 75)

artifacts = [
    ("Preprocessed Clean Dataset", clean_csv_path),
    ("Preprocessing Rejection Audit", rejected_csv_path),
    ("Preprocessing Summary Report", report_json_path),
    ("Computed Embeddings Dataset", diff_output_csv),
    ("Layer Geometry Statistics", stats_summary_path),
    ("2D PCA Projection Image", VIS_OUT_DIR / "2d_vector_projection.png"),
    ("3D PCA Projection Image", VIS_OUT_DIR / "3d_vector_projection.png"),
    ("Interactive 3D Plotly HTML", VIS_OUT_DIR / "interactive_3d_latent_space.html"),
]

for label, path in artifacts:
    exists = path.exists()
    status = "✓ FOUND" if exists else "✗ PENDING"
    size_str = f"({path.stat().st_size:,} bytes)" if exists else ""
    print(f"{status:<10} {label:<32} -> {path.name} {size_str}")

print("=" * 75)
print("Master Pipeline run completed successfully.")